# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

In [1]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

print("DuckDB initialized.")

DuckDB initialized.


In [2]:
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError
import os

token = os.environ.get("HF_TOKEN")

if not token:
    print("HF_TOKEN is missing from this Python environment.")
else:
    api = HfApi(token=token)

    try:
        user = api.whoami()
        print("Authentication successful.")
        print("Account:", user.get("name", "authenticated user"))

    except Exception as e:
        print("Authentication failed.")
        print("Error type:", type(e).__name__)
        print("Details:", str(e)[:500])

    try:
        api.dataset_info("FlyRank/internship-warehouse")
        print("Dataset access successful.")

    except Exception as e:
        print("\nDataset access failed.")
        print("Error type:", type(e).__name__)
        print("Details:", str(e)[:500])

Authentication successful.
Account: dheep8
Dataset access successful.


In [7]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ["HF_TOKEN"])

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for filename in files:
    if "fact_content_daily_performance" in filename:
        print(filename)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [8]:
import os
import duckdb
from huggingface_hub import hf_hub_url


if "con" not in globals():
    con = duckdb.connect()


con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

march_path = (
    "fact_content_daily_performance/"
    "month=2026-03/data_0.parquet"
)

march_url = hf_hub_url(
    repo_id="FlyRank/internship-warehouse",
    filename=march_path,
    repo_type="dataset"
)

print("March partition URL created.")
print("Path:", march_path)

March partition URL created.
Path: fact_content_daily_performance/month=2026-03/data_0.parquet


In [9]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing from this notebook's environment.")


con.execute("CREATE OR REPLACE SECRET hf_secret (TYPE http, BEARER_TOKEN ?)", [HF_TOKEN])

In [10]:
schema_df = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{march_url}') LIMIT 0"
).df()

display(schema_df)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

### Unit of analysis

One row represents one content item's daily performance for one client, identified by the report date, client, and content keys. The grain will be verified against the warehouse.

### Time window

March 2026 is the development month for the verification queries and feature frame. June 2026 is reserved as the sealed final month and will not be used to develop the label or tune features.

## 2. Fields: feature / label / context / excluded

### Field classification

**Features:** Historical search-performance metrics and content attributes that are available at the prediction decision time.

**Label / proxy:** The documented outcome used to identify declining content. The exact label definition must be verified against the warehouse schema and existing project framing.

**Context:** Client and content identifiers, report dates, and other fields used for grouping, joining, and time-based splitting. These are not automatically model features.

**Excluded:** `trend_direction`, `trend_pct`, and `is_declining_label` are excluded from the legitimate feature set because they describe or directly determine the decline outcome. Including them would leak the answer into the model.

In [12]:
query_slice = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS minimum_date,
    MAX(report_date) AS maximum_date
FROM read_parquet('{march_url}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""

slice_result = con.execute(query_slice).df()
display(slice_result)

,total_rows,minimum_date,maximum_date
0,9841378,2026-03-01,2026-03-31


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# One row per report_date x client_hash_id x content_hash_id.

query_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
        AS distinct_grain_keys,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
        AS duplicate_key_rows
FROM read_parquet('{march_url}')
"""

grain_result = con.execute(query_grain).df()
display(grain_result)


,total_rows,distinct_grain_keys,duplicate_key_rows
0,9841378,9841378,0


In [16]:
# QUERY 3: Verify Search Console data availability

query_availability = f"""
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_or_unknown_rows
FROM read_parquet('{march_url}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""

availability_result = con.execute(query_availability).df()
display(availability_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,available_rows,unavailable_or_unknown_rows
0,9841378,3611061,6230317


In [17]:
# Five-feature frame for March 2026.
# Only rows with confirmed Search Console availability are retained.

feature_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    -- Feature 1
    CAST(gsc_impressions AS DOUBLE) AS impressions,

    -- Feature 2
    CAST(gsc_clicks AS DOUBLE) AS clicks,

    -- Feature 3
    CASE
        WHEN gsc_impressions > 0
        THEN 100.0 * gsc_clicks / gsc_impressions
        ELSE NULL
    END AS ctr_pct,

    -- Feature 4
    gsc_avg_position AS avg_position,

    -- Feature 5
    CAST(gsc_sum_position AS DOUBLE) AS sum_position,

    -- Separate outcome proxy; not a model input
    CASE
        WHEN gsc_impressions > 0
        THEN 100.0 * gsc_clicks / gsc_impressions
        ELSE NULL
    END AS target_ctr_pct

FROM read_parquet('{march_url}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND gsc_data_available IS TRUE
"""

feature_df = con.execute(feature_query).df()

print("Feature frame shape:", feature_df.shape)
display(feature_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (3611061, 9)


,report_date,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,sum_position,target_ctr_pct
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20.0,0.0,0.0,3.350000,67.0,0.0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1.0,0.0,0.0,0.000000,0.0,0.0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125.0,1.0,0.8,4.928000,616.0,0.8
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7.0,0.0,0.0,4.000000,28.0,0.0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11.0,0.0,0.0,2.272727,25.0,0.0


### Feature availability

1. **Impressions:** Available after the relevant reporting period has closed and the Search Console data has been ingested.
2. **Clicks:** Available after the relevant reporting period has closed and the Search Console data has been ingested.
3. **CTR:** Available once clicks and impressions for the historical period are available; undefined when impressions are zero.
4. **Average position:** Available after the corresponding Search Console reporting data has been ingested.
5. **Sum of position:** Available after the corresponding Search Console reporting data has been ingested.

These features are candidates for historical inputs only. Their values must come from a period ending before the prediction cutoff. March metrics cannot be used to predict a March outcome.

In [18]:
# Leakage demonstration using a small, reproducible sample.
# The target is a proxy, not a claim about the official project label.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

leak_query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,
    100.0 * gsc_clicks / gsc_impressions AS target_ctr_pct
FROM read_parquet('{march_url}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND gsc_data_available IS TRUE
  AND gsc_impressions > 0
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
  AND gsc_sum_position IS NOT NULL
USING SAMPLE 10000 ROWS
"""

leak_df = con.execute(leak_query).df()

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_sum_position",
]

X = leak_df[features]
y = leak_df["target_ctr_pct"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
honest_mae = mean_absolute_error(y_test, model.predict(X_test))

# Deliberately leaked feature: the target itself.
X_leaked = X.copy()
X_leaked["LEAK_TARGET"] = y

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42
)

leaked_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

leaked_model.fit(X_train_l, y_train_l)
leaked_mae = mean_absolute_error(
    y_test_l, leaked_model.predict(X_test_l)
)

print("Honest MAE:", honest_mae)
print("Leaked MAE:", leaked_mae)
print("Lower MAE is better, but the leaked result is invalid.")

Honest MAE: 0.027083355935335656
Leaked MAE: 0.006009900886328902
Lower MAE is better, but the leaked result is invalid.


In [19]:
# Remove the deliberately leaked feature.
X_final = X.copy()

assert "LEAK_TARGET" not in X_final.columns

retained_honest_mae = honest_mae

print("Leaking feature removed:", "LEAK_TARGET" not in X_final.columns)
print("Retained honest MAE:", retained_honest_mae)
print("Retained feature count:", X_final.shape[1])

Leaking feature removed: True
Retained honest MAE: 0.027083355935335656
Retained feature count: 4


## 4. Data limits

### Limitation: incomplete Search Console coverage

The March 2026 warehouse partition contains 9,841,378 rows, but only 3,611,061 rows have `gsc_data_available IS TRUE`. This means that most rows in the partition cannot be treated as confirmed Search Console observations.

The analysis is therefore limited to the subset with confirmed availability. Its findings may not generalize to content items or clients without usable Search Console data. Missing or unavailable observations must not be interpreted as zero clicks or impressions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
